### Short Term Reversal Trading strategy

Strategy where worst performing asset of prior month is bought and hold them for one month.

We will consider the S&P100 stocks
- Buy the previous month loser stocks
- Hold them for one month
- Afterthat, we will backtest that over the last 12 years and compare that to S&P100 benchmark performance

In [1]:
import pandas as pd 
import datetime as dt 
import yfinance as yf 
from pandas.tseries.offsets import MonthEnd 

In [2]:
tickers = pd.read_html('https://en.wikipedia.org/wiki/S%26P_100')[2]
tickers 

,Symbol,Name,Sector
0,AAPL,Apple,Information Technology
1,ABBV,AbbVie,Health Care
2,ABT,Abbott Laboratories,Health Care
3,ACN,Accenture,Information Technology
4,ADBE,Adobe,Information Technology
...,...,...,...
96,V,Visa,Information Technology
97,VZ,Verizon,Communication Services
98,WFC,Wells Fargo,Financials
99,WMT,Walmart,Consumer Staples


We only need the symbols. Redefining tickers dataframe,

In [3]:
tickers = tickers.Symbol.to_list()

In [4]:
df = yf.download(tickers, start='2010-01-01') 

[*********************100%%**********************]  101 of 101 completed

1 Failed download:
['BRK.B']: YFTzMissingError('$%ticker%: possibly delisted; No timezone found')


In [5]:
prices = df['Adj Close']

Transforming string formated Date (index column) to datetime format

In [6]:
prices.index = pd.to_datetime(prices.index)  

Calculating monthly returns

In [7]:
mtl_ret = prices.pct_change().resample('M').agg(lambda x: (x+1).prod()-1)

In [8]:
mtl_ret.head() 

Ticker,AAPL,ABBV,ABT,ACN,ADBE,AIG,AMD,AMGN,AMT,AMZN,...,TXN,UNH,UNP,UPS,USB,V,VZ,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-31,-0.102565,0.0,-0.020811,-0.025672,-0.129145,-0.189361,-0.230928,0.013167,-0.023239,-0.063406,...,-0.130514,0.046622,-0.074782,-0.007047,0.096154,-0.069321,-0.103209,0.040629,-0.014751,-0.068258
2010-02-28,0.065396,0.0,0.025311,-0.024884,0.072755,0.022287,0.060322,-0.031976,0.004947,-0.055897,...,0.083556,0.026061,0.118101,0.025234,-0.018740,0.041183,-0.016655,-0.036666,0.011978,0.015429
2010-03-31,0.148471,0.0,-0.029477,0.049537,0.020779,0.378280,0.171934,0.057057,-0.001172,0.146706,...,0.003691,-0.035145,0.088022,0.096527,0.053628,0.067425,0.072243,0.138258,0.034093,0.030462
2010-04-30,0.111022,0.0,-0.020672,0.049417,-0.050042,0.139426,-0.021575,-0.042280,-0.042244,0.009796,...,0.067763,-0.071385,0.032196,0.073436,0.034390,-0.008788,-0.053940,0.063946,-0.035252,0.011795
2010-05-31,-0.016125,0.0,-0.070368,-0.140238,-0.045238,-0.090488,-0.055127,-0.096493,-0.006861,-0.084902,...,-0.061130,-0.040911,-0.051342,-0.085981,-0.104968,-0.195747,-0.047751,-0.132177,-0.051944,-0.101806


Calculating the returns of stocks of prior month of the formation date

In [9]:
formation = dt.datetime(2010,2,28)  

In [10]:
formation 

datetime.datetime(2010, 2, 28, 0, 0)

In [11]:
ret_1 = mtl_ret.loc[formation-MonthEnd(1)]

In [12]:
ret_1 

Ticker
AAPL   -0.102565
ABBV    0.000000
ABT    -0.020811
ACN    -0.025672
ADBE   -0.129145
          ...   
V      -0.069321
VZ     -0.103209
WFC     0.040629
WMT    -0.014751
XOM    -0.068258
Name: 2010-01-31 00:00:00, Length: 101, dtype: float64

In [13]:
ret_1.name='prev_month'

In [14]:
ret_1 = pd.DataFrame(ret_1)

In [15]:
ret_1

,prev_month
Ticker,
AAPL,-0.102565
ABBV,0.000000
ABT,-0.020811
ACN,-0.025672
ADBE,-0.129145
...,...
V,-0.069321
VZ,-0.103209
WFC,0.040629


In [16]:
ret_1['decile'] = pd.qcut(ret_1.prev_month,10,labels=False)

In [17]:
ret_1.head() 

,prev_month,decile
Ticker,,
AAPL,-0.102565,1
ABBV,0.000000,7
ABT,-0.020811,6
ACN,-0.025672,6
ADBE,-0.129145,1


All losers are having decile=0

In [18]:
losers = ret_1[ret_1.decile==0].index 

In [19]:
losers 

Index(['AIG', 'AMD', 'CRM', 'GOOG', 'GOOGL', 'GS', 'MS', 'NVDA', 'QCOM',
       'TMUS', 'TXN'],
      dtype='object', name='Ticker')

In [20]:
loser_ret = mtl_ret.loc[formation,mtl_ret.columns.isin(losers)].mean() 

In [21]:
loser_ret 

0.03801241545194156

In [22]:
def reversal(formation):
    ret_l = mtl_ret.loc[formation-MonthEnd(1)] 
    ret_l.name = 'mtl_return'
    ret_l = pd.DataFrame(ret_l)
    ret_l['decile'] = pd.qcut(ret_l.mtl_return,10,labels=False,duplicates='drop')
    losers = ret_l[ret_l.decile==0].index #To get index symbols of losers
    loser_ret =  mtl_ret.loc[formation,mtl_ret.columns.isin(losers)].mean() 
    return loser_ret 

In [23]:
formation 

datetime.datetime(2010, 2, 28, 0, 0)

In [24]:
reversal(formation) 

0.03801241545194156

Since, formation date is in index 1, we can start at index 2

In [25]:
returns = []
dates = []

for i in mtl_ret.index[2:]:
    returns.append(reversal(i)) 
    dates.append(i) 

In [26]:
frame = pd.DataFrame({'dates':dates,'returns':returns})

In [27]:
frame.head()  

,dates,returns
0,2010-03-31,0.083804
1,2010-04-30,-0.025496
2,2010-05-31,-0.073369
3,2010-06-30,-0.076613
4,2010-07-31,0.077027


In [28]:
frame['returns'].mean() 

0.016403093909129994

In [29]:
df2 = yf.download('^OEX',start='2010-01-01')['Adj Close']

[*********************100%%**********************]  1 of 1 completed


In [30]:
df2.head()  

Date
2010-01-04    522.729980
2010-01-05    524.599976
2010-01-06    524.309998
2010-01-07    526.340027
2010-01-08    527.760010
Name: Adj Close, dtype: float64

Resampling the returns on a monthly basis

In [31]:
bench_ret = df2.pct_change().resample('M').agg(lambda x:(x+1).prod()-1)

We have started form index 2 above. So, here also, we have to create a dataframe where the bench_ret remove the first 2 values

In [32]:
frame['S&P100 returns'] = bench_ret[2:].values 

In [33]:
frame.head()  

,dates,returns,S&P100 returns
0,2010-03-31,0.083804,0.057009
1,2010-04-30,-0.025496,0.010039
2,2010-05-31,-0.073369,-0.087399
3,2010-06-30,-0.076613,-0.051535
4,2010-07-31,0.077027,0.070373


We have to analyse how often the short term reversal strategy developed outperformed the S&P100 returns

In [34]:
frame[frame['returns'] > frame['S&P100 returns']].shape 

(102, 3)

In [35]:
frame.shape 

(172, 3)

Here, out of 172 returns, 102 times the strategy developed outperformed S&P100 returns

In [36]:
102/172

0.5930232558139535